# Data Visualization and Summary Measures

**STA 532 2.0 Spreadsheet Modelling** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta532.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Choose the variable scale](#theory-01)
- [Numerical summaries](#theory-02)
- [Visual choices](#theory-03)
- [Outlier detection](#theory-04)
- [Relationships between variables](#theory-05)
- [Pivot tables](#theory-06)
- [Weighted aggregates](#theory-07)
- [Dashboard construction](#theory-08)
- [Straight-line modelling in spreadsheets](#theory-09)

## Detailed theory · Summary measures, visualization and dashboards

<a id="theory-01"></a>

### Choose the variable scale
Categorical variables need category counts or proportions; quantitative variables can be summarized by location, spread and shape. A frequency table is a model of the question's denominator. Nominal category codes do not justify means, and ordinal codes do not guarantee equal distances.

<a id="theory-02"></a>

### Numerical summaries
Mean, median and mode answer different location questions. Sample SD measures individual spread using n−1; SE estimates variability of an estimator. Minimum, maximum and quartiles provide context. A summary-data tool is not a substitute for checking types and missing values first.

<a id="theory-03"></a>

### Visual choices
Use a bar chart for categories, histogram for a numerical distribution, boxplot for group spread, scatterplot for two quantitative variables and a correctly ordered line chart for time. Equal bin heights mean equal frequencies only with equal widths. Label axes, units, periods, sample sizes and any transformed scales. Avoid visual effects that distort comparisons.

<a id="theory-04"></a>

### Outlier detection
IQR fences flag values below $Q_1-1.5IQR$ or above $Q_3+1.5IQR$. Standard-score rules depend on the distribution and the reference mean/SD and can be distorted by the very extremes they flag. A flag is an investigation prompt, not proof of an error. Verify source records and report sensitivity to any justified treatment.

<a id="theory-05"></a>

### Relationships between variables
Identify response and explanatory roles where relevant, plot first and then calculate Pearson correlation for linear association. Covariance retains units; correlation divides by both SDs. Nonlinearity, outliers, mixtures and restricted ranges can defeat a one-number description. Rank correlation addresses monotone association; correlation does not establish causation.

<a id="theory-06"></a>

### Pivot tables
A pivot groups rows into categories and applies an aggregation. Sum, count, mean and distinct count answer different questions. Check whether a field has been interpreted as text, whether missing categories are present and whether filters apply. Refresh a pivot after data changes and verify its source range.

<a id="theory-07"></a>

### Weighted aggregates
For profit margin, use total profit divided by total revenue when the question concerns aggregate margin. An unweighted mean of row percentages gives equal influence to rows with very different denominators. The same issue appears in mean prices, conversion rates and group percentages.

<a id="theory-08"></a>

### Dashboard construction
A dashboard brings selected measures together for a defined audience. Show the measure definition, time window, filters and baseline; use consistent colours and scales. Retain a table or calculation trail behind each headline value. A dynamic-looking chart does not compensate for an incorrect subtotal or changing denominator.

<a id="theory-09"></a>

### Straight-line modelling in spreadsheets
A fitted straight line summarizes a conditional response, not just a correlation. Report intercept, slope, units, fit and residual checks; use the regression notebook for coefficient inference and multiple-predictor details. Predictions outside the observed input combinations need additional justification.

## Concepts covered

**Course points:** Probability theory; binomial and Poisson; normal distribution; z score, CDF, tail probability and quantile.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Pivot-style summary: sums and weighted margins

### Question and assumptions

Summarize sales for two regions using revenue and cost totals.

### Procedure

1. Calculate row profit.
2. Group by region and sum revenue, cost and profit.
3. Divide total profit by total revenue for each region, rather than averaging row margins.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Region | Revenue | Cost | Profit | Row margin |
| --- | --- | --- | --- | --- |
| A | 100 | 70 | 30 | 0.3 |
| A | 1000 | 900 | 100 | 0.1 |
| B | 250 | 150 | 100 | 0.4 |
| B | 400 | 280 | 120 | 0.3 |

| Region | Count | Revenue sum | Cost sum | Profit sum | Aggregate margin |
| --- | --- | --- | --- | --- | --- |
| A | 2 | 1100 | 970 | 130 | 0.118182 |
| B | 2 | 650 | 430 | 220 | 0.338462 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


data=[["A",100,70],["A",1000,900],["B",250,150],["B",400,280]]
show_table(["Region","Revenue","Cost","Profit","Row margin"],[[g,r,c,r-c,(r-c)/r] for g,r,c in data])
rows=[]
for g in sorted(set(row[0] for row in data)):
    chosen=[row for row in data if row[0]==g]; r=sum(row[1] for row in chosen); c=sum(row[2] for row in chosen)
    rows.append([g,len(chosen),r,c,r-c,(r-c)/r])
show_table(["Region","Count","Revenue sum","Cost sum","Profit sum","Aggregate margin"],rows)
assert abs(rows[0][-1]-130/1100)<1e-12


### Decision and interpretation

Region A has a combined margin about 11.82%, not the arithmetic average of 30% and 10%. Charts should display the aggregate denominator and units.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
lam=6.5; x=rng.poisson(lam,100000)
print('Poisson mean/var theory',lam,lam,'sim',x.mean().round(2),x.var().round(2))
print('P(X<=8)',stats.poisson.cdf(8,lam).round(4),'P(X>=10)',stats.poisson.sf(9,lam).round(4))
print('Binomial P(defects>=4)',stats.binom.sf(3,30,.08).round(4))
mu,sd=52,7; print('z',((65-mu)/sd),'upper tail',stats.norm.sf(65,mu,sd).round(4),'95th percentile',stats.norm.ppf(.95,mu,sd).round(2))
plt.hist(x,bins=np.arange(20)-.5); plt.show()

### Practice set · Risk desk
1. For X~Binomial(30,.08), find E[X] and P(X≥4).
2. For Poisson rate 6.5/hour, compute P(X≥10) and the smallest staffing capacity exceeded no more than 5% of hours.
3. For N(52,7²), compute P(X>65) and the 95th percentile.
4. Identify one reason a Poisson model may fail for arrivals.

<details><summary>Answer sketches</summary>
1. Mean 2.4; use stats.binom.sf(3,30,.08). 2. Use stats.poisson.sf(9,6.5); threshold is the smallest k with CDF(k)≥.95. 3. z=(65−52)/7≈1.857; upper tail≈.0317; 95th percentile≈63.52. 4. Rates may change over time, events may cluster, or events may not be independent.
</details>

### Debrief
Use an upper-tail probability to choose a staffing threshold.

Record the formula, assumptions, and one follow-up question.